In [2]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

import numpy as np

In [3]:
import os
# save_dir = '/app/local_project/Nov21 single holes'
save_dir = '/Users/Howfishy/Documents/0. tidy3d/May10 all gaps'
os.makedirs(save_dir, exist_ok=True)

In [8]:
# Define each simulation parameter
prefix = "_2_dipolEz"
suffix = "_vertical_n1" + "_nophase" +"_in_out"  # not changed

# ==============================================================================
# PARAMETER NAMES (explicit)
# ==============================================================================
y_edge_G20     = 'y_edge'     + prefix  + "_Th100D70G20"  + suffix
y_edge_G50     = 'y_edge'     + prefix  + "_Th100D70G50"  + suffix
y_edge_G100    = 'y_edge'     + prefix  + "_Th100D70G100" + suffix

solid_edge_G20  = 'solid_edge' + prefix  + "_Th100D70G20"  + suffix
solid_edge_G50  = 'solid_edge' + prefix  + "_Th100D70G50"  + suffix
solid_edge_G100 = 'solid_edge' + prefix  + "_Th100D70G100" + suffix

close_edge_G20  = 'close_edge' + prefix  + "_Th100D70G20"  + suffix
close_edge_G50  = 'close_edge' + prefix  + "_Th100D70G50"  + suffix
close_edge_G100 = 'close_edge' + prefix  + "_Th100D70G100" + suffix

center_G20  = 'center'     + prefix  + "_Th100D70G20"  + suffix
center_G50  = 'center'     + prefix  + "_Th100D70G50"  + suffix
center_G100 = 'center'     + prefix  + "_Th100D70G100" + suffix

edge_G20  = 'edge'       + prefix  + "_Th100D70G20"  + suffix
edge_G50  = 'edge'       + prefix  + "_Th100D70G50"  + suffix
edge_G100 = 'edge'       + prefix  + "_Th100D70G100" + suffix

# ==============================================================================
# STYLES: color encodes probe position, dash encodes gap size
# ==============================================================================
param_styles = {
    y_edge_G20     : dict(color='rgb(242, 117, 163)', dash='solid'),
    y_edge_G50     : dict(color='rgb(242, 117, 163)', dash='dash'),
    y_edge_G100    : dict(color='rgb(242, 117, 163)', dash='dot'),

    solid_edge_G20  : dict(color='rgb(0,   128, 128)', dash='solid'),
    solid_edge_G50  : dict(color='rgb(0,   128, 128)', dash='dash'),
    solid_edge_G100 : dict(color='rgb(0,   128, 128)', dash='dot'),

    close_edge_G20  : dict(color='rgb(4,   55,  242)', dash='solid'),
    close_edge_G50  : dict(color='rgb(4,   55,  242)', dash='dash'),
    close_edge_G100 : dict(color='rgb(4,   55,  242)', dash='dot'),

    center_G20  : dict(color='rgb(0,   0,   0  )', dash='solid'),
    center_G50  : dict(color='rgb(0,   0,   0  )', dash='dash'),
    center_G100 : dict(color='rgb(0,   0,   0  )', dash='dot'),

    edge_G20  : dict(color='rgb(102, 0,   204)', dash='solid'),
    edge_G50  : dict(color='rgb(102, 0,   204)', dash='dash'),
    edge_G100 : dict(color='rgb(102, 0,   204)', dash='dot'),
}

# ==============================================================================
# OFFSET GROUPS: row 0 = highest offset (top of plot and top of legend)
# ==============================================================================
y_edge_ALL     = [y_edge_G20,     y_edge_G50,     y_edge_G100    ]
solid_edge_ALL = [solid_edge_G20, solid_edge_G50, solid_edge_G100]
close_edge_ALL = [close_edge_G20, close_edge_G50, close_edge_G100]
center_ALL     = [center_G20,     center_G50,     center_G100    ]
edge_ALL       = [edge_G20,       edge_G50,       edge_G100      ]

offset_groups = [
    y_edge_ALL,      # row 0 → highest offset
    solid_edge_ALL,
    close_edge_ALL,
    center_ALL,
    edge_ALL,        # row 4 → lowest offset
]

# Definition

In [1]:
def plot_waterfall(
    fields,           # list[str] or one str e.g. ["Ez", "Hx"] or "Ez"
    monitors,         # list[str] or one str
    
    offset_groups,    # list[list[str]]: row 0 = highest offset, last row = lowest
                      #   e.g. [ [y_edge_G20, y_edge_G50, ...],   # row 0 → top
                      #          [solid_edge_G20, ...],            # row 1
                      #          [center_G20, ...] ]               # row 2 → bottom
    param_styles,     # dict[str, dict]: {param: dict(color='rgb(...)', dash='solid')}
    
    data_dir,         # str or Path: root directory containing data_<param>/ subfolders
    offset_factor=1.0,
    save_html=True,
    verbose=True,
):
    
    # --- normalise scalar inputs to lists ---
    if isinstance(fields, str):
        fields = [fields]
    if isinstance(monitors, str):
        monitors = [monitors]

    # --- Ensure param → group index (topmost offset = idx 0) ---
    param_to_group = {}
    for group_idx, group in enumerate(offset_groups):
        for param in group:
            param_to_group[param] = group_idx

    n_groups   = len(offset_groups)
    data_dir   = Path(data_dir)
    figs       = {}

    # Row 0 = highest offset = plotted last (drawn on top, appears at top of legend naturally)
    # Row (n-1) = lowest offset = plotted first
    params_bottom_first = []
    for group in reversed(offset_groups):   # reversed: last row first
        params_bottom_first.extend(group)

    # ==========================================================================
    for field in fields:
        for monitor in monitors:

            output_name = f"waterfall_{field}_{monitor}"
            fig = go.Figure()

            spectra_subdir = "spectraE" if field.startswith("E") else "spectraH"

            for param in params_bottom_first:

                file_path = (
                    data_dir / f"data_{param}" / spectra_subdir
                    / f"{field}_{monitor}_spectrum.csv"
                )

                if not file_path.exists():
                    if verbose:
                        print(f"  [missing] {file_path}")
                    continue

                df     = pd.read_csv(file_path)
                x_data = df.iloc[:, 0].values
                y_data = df.iloc[:, 1].values

                # offset: row 0 → (n_groups-1)*offset_factor (highest)
                #         row i → (n_groups-1-i)*offset_factor
                group_idx = param_to_group[param]
                y_offset  = (n_groups - 1 - group_idx) * offset_factor
                y_plot    = y_data + y_offset

                style = param_styles.get(param, dict(color="rgb(0,0,0)", dash="solid"))

                fig.add_trace(go.Scatter(
                    x=x_data,
                    y=y_plot,
                    mode="lines",
                    name=param,
                    line=dict(
                        color=style.get("color", "rgb(0,0,0)"),
                        dash=style.get("dash",  "solid"),
                        width=style.get("width", 1.5),
                    ),
                    showlegend=True,
                ))

            fig.update_layout(
                title=f"Waterfall: {field} — {monitor}",
                xaxis_title="Energy (eV)",
                yaxis_title="Spectrum (offset)",
                hovermode="x unified",
                yaxis=dict(
                    showticklabels=False,
                    showgrid=True,
                    zeroline=False,
                    visible=True,
                ),
                legend=dict(
                    traceorder="normal",  # traces added bottom-first, so legend reads top-group first naturally
                ),
            )

            if save_html:
                out_path = data_dir / f"{output_name}.html"
                fig.write_html(str(out_path))
                if verbose:
                    print(f"Saved: {out_path}")

            figs[(field, monitor)] = fig

    print(f"\n✓ Generated {len(figs)} plot(s)")
    return figs

In [6]:
Monitors = [
    'DFT_in_plane_slice0',
    # 'DFT_in_plane_slice1',
    # 'DFT_in_plane_slice2',
    # 'DFT_in_plane_slice3',
    'DFT_in_plane_slice4.5',
  
    'DFT_out_plane',
]


####### Working histroy ############
# 'DFT_out_plane',
# 'DFT_out_plane_XZoffset',

# 'DFT_out_plane_YZ',
# 'DFT_out_plane_YZoffset',

# 'DFT_bottom_plane_slice0.5',
# 'DFT_bottom_plane_slice2',
# 'DFT_bottom_plane_slice4',
# 'DFT_bottom_plane_slice8',
# 'DFT_bottom_plane_slice14',

# 'DFT_in_plane_centre',
# 'DFT_in_plane_bottom',
# 'DFT_in_plane_centre_OFFSET',
# 'DFT_in_plane_bottom_OFFSET',

    # 'DFT_in_plane_slice0_Left',
    # 'DFT_in_plane_slice0_Right',

In [9]:
Fields = [
    "Ez",     
    # "Ex", 
    # "Ey", 

    "Hx",
    # "Hy",
    # "Hz"
]

figs = plot_waterfall(
    fields        = Fields,
    monitors      = Monitors,
    offset_groups = offset_groups,
    param_styles  = param_styles,
    data_dir      = save_dir,
    offset_factor = 1.0,
)

  [missing] \Users\Howfishy\Documents\0. tidy3d\May10 all gaps\data_edge2_dipolEz_Th100D70G20_vertical_n1_nophase_in_out\spectraE\Ez_DFT_in_plane_slice0_spectrum.csv
  [missing] \Users\Howfishy\Documents\0. tidy3d\May10 all gaps\data_edge2_dipolEz_Th100D70G50_vertical_n1_nophase_in_out\spectraE\Ez_DFT_in_plane_slice0_spectrum.csv
  [missing] \Users\Howfishy\Documents\0. tidy3d\May10 all gaps\data_edge2_dipolEz_Th100D70G100_vertical_n1_nophase_in_out\spectraE\Ez_DFT_in_plane_slice0_spectrum.csv
  [missing] \Users\Howfishy\Documents\0. tidy3d\May10 all gaps\data_center2_dipolEz_Th100D70G20_vertical_n1_nophase_in_out\spectraE\Ez_DFT_in_plane_slice0_spectrum.csv
  [missing] \Users\Howfishy\Documents\0. tidy3d\May10 all gaps\data_center2_dipolEz_Th100D70G50_vertical_n1_nophase_in_out\spectraE\Ez_DFT_in_plane_slice0_spectrum.csv
  [missing] \Users\Howfishy\Documents\0. tidy3d\May10 all gaps\data_center2_dipolEz_Th100D70G100_vertical_n1_nophase_in_out\spectraE\Ez_DFT_in_plane_slice0_spectrum.